# backward-fn-signature — ex3: multiply_back0 and multiply_back1 with broadcast-reduce

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backward-fn-signature`. Running the final beacon cell reports progress against the `Backprop: backward fn signature` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backward fn signature` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-fn-signature`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-fn-signature"
DD_SUBTOPIC = "Backprop: backward fn signature"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## backward fn signature — quick refresher

The ARENA back-fn convention is uniform across unary and binary ops:
```python
back_fn(grad_out, out, *args, **kwargs) -> grad_in
```
For a BINARY op like `multiply`, the `*args` slot is `(x, y)`, and you write TWO back fns — `multiply_back0` (gradient wrt `x`) and `multiply_back1` (gradient wrt `y`).

**This drill (ex3) vs prior.** ex1 wrote `log_back` (unary). ex2 wrote `negative_back` + `exp_back` (both unary). ex3 is the first BINARY back-fn drill — `y` enters the signature, AND broadcasting means the raw chain-rule output can have a different shape than the input you're differentiating against, so you have to sum over the broadcast axes.

### Exercise 3 — multiply_back0 and multiply_back1 with broadcast-reduce

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the (grad_out, out, x, y) back-fn signature to write multiply_back0 and multiply_back1 so that each grad has the same shape as its input — summing over broadcast axes when needed.
> Keywords: multiply-back, binary-op, broadcast-reduce, grad-shape-match
> ```

**KCs targeted:** `backward-fn-signature`, `back-fn-broadcast-axis-sum`

Implement TWO backward fns for `out = x * y` with the extended binary signature.

**1. `multiply_back0(grad_out, out, x, y) -> grad_x`** — gradient wrt `x`.
   - Math: `d(x*y)/dx = y`, so raw `grad_x = grad_out * y`.
   - If `x` and `y` broadcast (e.g. `x: (3,)`, `y: (4, 3)`), then `grad_out` has shape `(4, 3)` and the raw product has shape `(4, 3)` — but `grad_x` MUST have shape `(3,)`. Sum over the broadcast axes (the leading axes of `grad_out` that aren't in `x`).

**2. `multiply_back1(grad_out, out, x, y) -> grad_y`** — mirror image. Raw `grad_y = grad_out * x`; sum over the broadcast axes of `y`.

**Use this helper** (provided in the test cell):
```python
def unbroadcast(grad, target_shape):
    # sum extra leading dims
    while grad.ndim > len(target_shape):
        grad = grad.sum(dim=0)
    # sum dims where target was size 1
    for i, s in enumerate(target_shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad
```

**Return** tensors with EXACTLY `x.shape` and `y.shape` respectively. The test verifies non-broadcasted, leading-broadcasted, and size-1-broadcasted cases.

In [ ]:
def multiply_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """∂L/∂x for out = x * y, broadcasting-aware."""
    raise NotImplementedError()


def multiply_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """∂L/∂y for out = x * y, broadcasting-aware."""
    raise NotImplementedError()


def _test_ex3():
    def unbroadcast(grad, target_shape):
        while grad.ndim > len(target_shape):
            grad = grad.sum(dim=0)
        for i, s in enumerate(target_shape):
            if s == 1 and grad.shape[i] != 1:
                grad = grad.sum(dim=i, keepdim=True)
        return grad

    # === Case A: no broadcasting — shapes match ===
    x = t.tensor([1.0, 2.0, 3.0])
    y = t.tensor([4.0, 5.0, 6.0])
    out = x * y
    grad_out = t.tensor([1.0, 1.0, 1.0])
    gx = multiply_back0(grad_out, out, x, y)
    gy = multiply_back1(grad_out, out, x, y)
    assert gx.shape == x.shape, f'gx shape {tuple(gx.shape)} != {tuple(x.shape)}'
    assert gy.shape == y.shape
    assert t.allclose(gx, y), f'gx should equal y when grad_out is ones: got {gx}'
    assert t.allclose(gy, x), f'gy should equal x when grad_out is ones: got {gy}'

    # === Case B: x is (3,), y is (4, 3) — broadcast leading axis ===
    x = t.tensor([1.0, 2.0, 3.0])               # (3,)
    y = t.tensor([[1.0]*3, [2.0]*3, [3.0]*3, [4.0]*3])  # (4, 3)
    out = x * y                                 # (4, 3)
    grad_out = t.ones(4, 3)
    gx = multiply_back0(grad_out, out, x, y)
    gy = multiply_back1(grad_out, out, x, y)
    assert gx.shape == x.shape, f'gx broadcasted shape {tuple(gx.shape)} != {tuple(x.shape)}'
    assert gy.shape == y.shape, f'gy shape {tuple(gy.shape)} != {tuple(y.shape)}'
    # gx[i] = sum over leading axis of y → 1+2+3+4 = 10 for every i.
    assert t.allclose(gx, t.tensor([10.0, 10.0, 10.0])), f'gx mismatch: {gx}'
    # gy[k, i] = grad_out[k,i] * x[i] = x[i].
    expected_gy = x.expand_as(y).clone()
    assert t.allclose(gy, expected_gy), f'gy mismatch: {gy}'

    # === Case C: x is (3, 1), y is (3, 4) — broadcast size-1 axis ===
    x = t.tensor([[1.0], [2.0], [3.0]])         # (3, 1)
    y = t.tensor([[1.0, 2.0, 3.0, 4.0]] * 3)    # (3, 4)
    out = x * y                                 # (3, 4)
    grad_out = t.ones(3, 4)
    gx = multiply_back0(grad_out, out, x, y)
    gy = multiply_back1(grad_out, out, x, y)
    assert gx.shape == x.shape, f'gx shape {tuple(gx.shape)} != (3,1)'
    assert gy.shape == y.shape, f'gy shape {tuple(gy.shape)} != (3,4)'
    # gx[i, 0] = sum over the size-1-broadcasted axis of grad_out * y = sum(y[i, :])
    #         = 1+2+3+4 = 10.
    assert t.allclose(gx, t.tensor([[10.0], [10.0], [10.0]])), f'gx mismatch: {gx}'

    # === Cross-check vs autograd ===
    xa = t.tensor([1.0, 2.0, 3.0], requires_grad=True)
    ya = t.tensor([[1.0]*3, [2.0]*3, [3.0]*3, [4.0]*3], requires_grad=True)
    (xa * ya).sum().backward()
    gx_ref = xa.grad
    gy_ref = ya.grad
    gx_ours = multiply_back0(t.ones_like(xa.expand_as(ya).contiguous() * ya),
                              xa * ya, xa.detach(), ya.detach())
    gy_ours = multiply_back1(t.ones_like(xa.expand_as(ya).contiguous() * ya),
                              xa * ya, xa.detach(), ya.detach())
    assert t.allclose(gx_ours, gx_ref, atol=1e-5), f'gx vs autograd: ours={gx_ours} ref={gx_ref}'
    assert t.allclose(gy_ours, gy_ref, atol=1e-5), f'gy vs autograd: ours={gy_ours} ref={gy_ref}'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def _unbroadcast(grad, target_shape):
    while grad.ndim > len(target_shape):
        grad = grad.sum(dim=0)
    for i, s in enumerate(target_shape):
        if s == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def multiply_back0(grad_out, out, x, y):
    raw = grad_out * y
    return _unbroadcast(raw, x.shape)

def multiply_back1(grad_out, out, x, y):
    raw = grad_out * x
    return _unbroadcast(raw, y.shape)
```

**Why binary ops need TWO back fns.** A binary op has two input positions in the graph (argnum 0 and 1), and each needs its own gradient calculation. The ARENA `BackwardFuncLookup` keys back fns by `(forward_fn, argnum)` to keep them separate.

**Why broadcast-aware reduction is essential.** PyTorch's `out = x * y` happily broadcasts `x: (3,)` with `y: (4, 3)`. The raw chain-rule output `grad_out * y` then has shape `(4, 3)` — but `grad_x` MUST have `x.shape == (3,)` for the optimizer to apply it. Summing over the broadcast axes restores the input shape. PyTorch's autograd does this internally; we have to do it explicitly.

**The `_unbroadcast` helper.** Two passes: (1) sum the leading dims that don't exist on the target, (2) sum dims where the target has size 1. Order matters — leading-dim reduction first reduces ndim; the size-1 pass then matches by position.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()